# Harvey-LLM Fine-Tuning (Unsloth + Qwen2.5-7B-Instruct + QLoRA)

Fine-tune **Qwen2.5-7B-Instruct** as a Harvey Specter conversational model using **[Unsloth](https://github.com/unslothai/unsloth)** for 2× faster QLoRA training and TRL's **SFTTrainer**.

**Runtime:** Google Colab with **T4 GPU** (16 GB VRAM)

**Before running:**
1. Upload `data/final/train.jsonl` and `data/final/val.jsonl` to Google Drive, or clone this repo
2. (Optional) Add your Hugging Face token for pushing the model

In [ ]:
# @title 1. Install Unsloth (Colab-optimized)
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title 2. Configuration
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    # Unsloth pre-quantized model (recommended over raw HF weights)
    base_model: str = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

    # Dataset paths — adjust after mounting Drive or cloning repo
    train_file: str = "data/final/train.jsonl"
    val_file: str = "data/final/val.jsonl"

    # Sequence length & quantization
    max_seq_length: int = 2048
    load_in_4bit: bool = True

    # QLoRA (Unsloth-optimized defaults)
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0  # Unsloth optimizes with dropout=0

    # Training (T4-friendly)
    num_epochs: int = 2
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4  # effective batch size = 8
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps: int = 10
    save_steps: int = 100
    eval_steps: int = 100
    seed: int = 3407

    # Output
    output_dir: str = "outputs/harvey-llm-qwen2.5-7b-lora"
    hf_repo: str = ""  # e.g. "your-username/harvey-llm" — leave empty to skip push

cfg = TrainingConfig()
print(cfg)

In [ ]:
# @title 3. Clone repo & verify dataset
import os

REPO_URL = "https://github.com/regional-specter/harvey-llm.git"

if not os.path.exists("harvey-llm"):
    !git clone {REPO_URL}
os.chdir("harvey-llm")

# --- Option B: Google Drive (if you uploaded the repo manually) ---
# from google.colab import drive
# drive.mount("/content/drive")
# os.chdir("/content/drive/MyDrive/harvey-llm")

assert os.path.exists(cfg.train_file), f"Missing {cfg.train_file}"
assert os.path.exists(cfg.val_file), f"Missing {cfg.val_file}"
print(f"Working directory: {os.getcwd()}")
print("Dataset files found.")

In [ ]:
# @title 4. Load dataset
import json
from datasets import Dataset

def load_jsonl(path: str) -> list[dict]:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(cfg.train_file)
val_data = load_jsonl(cfg.val_file)

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples:   {len(val_dataset)}")
print("\nSample:")
print(json.dumps(train_dataset[0], indent=2)[:500])

In [ ]:
# @title 5. Load model with Unsloth (4-bit QLoRA-ready)
from unsloth import FastLanguageModel
import torch

dtype = None  # Unsloth auto-detects (bf16 on Ampere+, fp16 on T4)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.base_model,
    max_seq_length=cfg.max_seq_length,
    dtype=dtype,
    load_in_4bit=cfg.load_in_4bit,
)

# GPU memory baseline (for Unsloth stats at end)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"Reserved memory = {start_gpu_memory} GB.")

In [ ]:
# @title 6. Apply LoRA adapters (Unsloth-optimized)
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_r,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM vs True
    random_state=cfg.seed,
)
model.print_trainable_parameters()

In [ ]:
# @title 7. Format dataset with Qwen chat template
from unsloth.chat_templates import get_chat_template

# Ensure Qwen2.5 chat template is applied
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts, batched=True)
val_dataset = val_dataset.map(formatting_prompts, batched=True)

print("Sample formatted example:")
print(train_dataset[0]["text"][:600])

In [ ]:
# @title 8. Configure SFTTrainer & train
import os
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.max_seq_length,
    packing=False,
    args=SFTConfig(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        per_device_eval_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        logging_steps=cfg.logging_steps,
        save_steps=cfg.save_steps,
        eval_strategy="steps",
        eval_steps=cfg.eval_steps,
        save_total_limit=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        seed=cfg.seed,
        report_to="none",
    ),
)

print("Starting Unsloth training...")
trainer_stats = trainer.train()
print("Training complete.")

In [ ]:
# @title 9. Training stats & save LoRA adapter
import os
import torch

# Memory / time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s training ({trainer_stats.metrics['train_runtime']/60:.2f} min)")
print(f"Peak reserved memory = {used_memory} GB ({used_percentage}%)")
print(f"Peak LoRA training memory = {used_memory_for_lora} GB ({lora_percentage}%)")

adapter_dir = os.path.join(cfg.output_dir, "final_adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"LoRA adapter saved to {adapter_dir}")

# --- Optional: merge LoRA into 16-bit for easier deployment ---
# merged_dir = os.path.join(cfg.output_dir, "merged_16bit")
# model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
# print(f"Merged 16-bit model saved to {merged_dir}")

# --- Optional: export to GGUF for Ollama / llama.cpp ---
# model.save_pretrained_gguf(os.path.join(cfg.output_dir, "gguf"), tokenizer, quantization_method="q4_k_m")

In [ ]:
# @title 10. Quick inference test (Unsloth 2× faster)
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

HARVEY_SYSTEM = (
    "You are Harvey Specter from the TV show Suits — a brilliant, confident, "
    "sharp-tongued corporate lawyer at Pearson Hardman (later Specter Litt). "
    "You speak with wit, arrogance, and precision. You never show weakness, "
    "you win every argument, and you deliver punchy one-liners. "
    "Stay in character at all times."
)

test_prompts = [
    "Harvey, I think we're going to lose this case.",
    "Someone said you're all style and no substance.",
    "Why do you always have to be the smartest person in the room?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": HARVEY_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    print(f"USER: {prompt}")
    print("HARVEY: ", end="")
    _ = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        streamer=TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True),
    )
    print()

In [ ]:
# @title 11. (Optional) Push to Hugging Face Hub
from huggingface_hub import login

if cfg.hf_repo:
    login()  # paste your HF token when prompted
    model.push_to_hub(cfg.hf_repo, token=True)
    tokenizer.push_to_hub(cfg.hf_repo, token=True)
    print(f"Pushed to https://huggingface.co/{cfg.hf_repo}")
else:
    print("Set cfg.hf_repo to push the model to Hugging Face.")